# 10 Does the feature ranking limit top-N? (side test)
For a few years per region the bands are ranked on **that year's train pixels**, then the classifier is scored on that year's val pixels with the top N bands.
Compared with: the pooled ranking used in the study (notebook 04), random bands of the same size, a per-year ranking with correlated bands removed, and the study's pruned set (06).
One fixed classifier (100 trees, min leaf 2) and the same training draw for every method. Ranking never sees val pixels.
Reading the result: if ranked beats random at small N and all curves meet at the same plateau, the plateau comes from redundant bands, not from the ranking.

In [ ]:
import os, json
from pathlib import Path
import numpy as np, pandas as pd
import spatial_split as ss
import feature_rank_check as fr

REGIONS = os.environ.get('REGIONS', 'HIM,PEN,DES,IGP,WGR,ICR,NEI').split(',')
N_YEARS = int(os.environ.get('N_YEARS_CHECK', 4))      # years per region (evenly spread over the search years)
N_RANDOM = int(os.environ.get('N_RANDOM', 4))          # random band sets per N
SEEDS = (0,)
N_JOBS = int(os.environ.get('RF_JOBS', 4))
artifact_dir = Path('data/feature_selection_artifacts'); out_dir = Path('data/feature_ranking_check'); out_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
summary = {}
for region in REGIONS:
    print(f"\n{'='*60}\n{region}\n{'='*60}")
    rk = json.load(open(artifact_dir / f'{region}_ranked_features.json'))
    all_ranked = rk['ranked_features']
    pooled_imp = np.array([rk['feature_importances'][f] for f in all_ranked])
    uncorr = json.load(open(artifact_dir / f'{region}_uncorrelated_features.json'))['uncorrelated_features']
    sy = ss.search_years(region)
    years = [sy[int(i)] for i in np.unique(np.linspace(0, len(sy) - 1, N_YEARS).round().astype(int))]
    scored = ss.scored_classes(region)[0]
    print(f'years {years}; {len(scored)} scored classes; {len(all_ranked)} bands')
    load = lambda r, y, cols: ss.load_year_split(r, y, cols, gated=True)
    sc, ag = fr.check_region(region, years, all_ranked, load, scored, pooled_imp=pooled_imp, uncorr=uncorr,
                             n_random=N_RANDOM, seeds=SEEDS, n_jobs=N_JOBS)
    sc.to_csv(out_dir / f'{region}_ranking_check.csv', index=False)
    ag.to_csv(out_dir / f'{region}_ranking_agreement.csv', index=False)
    m, res = fr.summarize(sc)
    res['mean_year_vs_year_top20_jaccard'] = ag.attrs.get('mean_year_vs_year_top20_jaccard', np.nan)
    res['mean_top20_jaccard_vs_pooled'] = ag.top20_jaccard_vs_pooled.mean()
    summary[region] = res
    fr.plot_region(sc, region, out_dir / f'{region}_ranking_check.png')
    print(m.round(4).to_string()); print(ag.round(3).to_string(index=False)); print(res)
pd.DataFrame(summary).T.to_csv(out_dir / 'summary_all_regions.csv')
print(pd.DataFrame(summary).T.round(3).to_string())